# 10 - Model Evaluation Results

This notebook consolidates the results from the ML phase of VisionInspect AI.

It combines:

- baseline defect detection results
- Anomalib PaDiM anomaly detection results
- defect classification results
- severity scoring examples
- final model artifact summary

This is the notebook that turns the experiments into report-ready evidence.

## Project Status

At this point, the AI/ML foundation is complete:

```text
Dataset -> Preprocessing -> Baseline -> Anomalib model -> Classifier -> Severity scoring
```

The next phase after this notebook is:

```text
FastAPI backend -> MongoDB -> Cloudinary -> Next.js frontend
```

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
CHART_DIR = PROJECT_ROOT / "outputs" / "charts"
MODELS_DIR = PROJECT_ROOT / "models"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Reports:", REPORT_DIR)
print("Charts:", CHART_DIR)

## Load Result Files

In [ ]:
def load_json(path):
    path = Path(path)
    if not path.exists():
        print("Missing:", path)
        return {}
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


baseline_metrics = load_json(REPORT_DIR / "baseline_detection_metrics.json")
anomalib_metrics = load_json(REPORT_DIR / "anomalib_padim_metrics.json")
classifier_metrics = load_json(REPORT_DIR / "defect_classifier_metrics.json")
model_metadata = load_json(MODELS_DIR / "model_metadata.json")

print("Baseline metrics keys:", list(baseline_metrics.keys()))
print("Anomalib metrics keys:", list(anomalib_metrics.keys()))
print("Classifier metrics keys:", list(classifier_metrics.keys()))

## Model Artifacts

Confirm the trained artifacts exist.

In [ ]:
artifact_rows = []

artifact_paths = {
    "PaDiM checkpoint": MODELS_DIR / "checkpoints" / "padim_mvtec_bottle_v1.ckpt",
    "Defect classifier": MODELS_DIR / "defect_classifier.pkl",
    "Model metadata": MODELS_DIR / "model_metadata.json",
}

for name, path in artifact_paths.items():
    artifact_rows.append({
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 2) if path.exists() else None,
    })

artifact_df = pd.DataFrame(artifact_rows)
artifact_df

## Baseline vs PaDiM Comparison

The baseline is a classical image-processing reference model.

PaDiM is the professional anomaly detection model.

In [ ]:
comparison_rows = []

if baseline_metrics:
    comparison_rows.append({
        "model": "Baseline Difference Model",
        "accuracy": baseline_metrics.get("accuracy"),
        "precision": baseline_metrics.get("precision"),
        "recall": baseline_metrics.get("recall"),
        "f1_score": baseline_metrics.get("f1_score"),
        "roc_auc": baseline_metrics.get("roc_auc"),
    })

if anomalib_metrics:
    comparison_rows.append({
        "model": "Anomalib PaDiM",
        "accuracy": None,
        "precision": None,
        "recall": None,
        "f1_score": anomalib_metrics.get("image_F1Score"),
        "roc_auc": anomalib_metrics.get("image_AUROC"),
        "pixel_auroc": anomalib_metrics.get("pixel_AUROC"),
        "pixel_f1": anomalib_metrics.get("pixel_F1Score"),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

In [ ]:
plot_data = comparison_df.set_index("model")[["f1_score", "roc_auc"]]

plot_data.plot(kind="bar", figsize=(9, 4))
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Baseline vs PaDiM Image-Level Performance")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()

comparison_chart_path = CHART_DIR / "baseline_vs_padim_comparison.png"
plt.savefig(comparison_chart_path, dpi=160)
plt.show()

print("Saved chart:", comparison_chart_path)

## PaDiM Pixel-Level Metrics

Pixel metrics evaluate localization/heatmap quality.

In [ ]:
padim_pixel_df = pd.DataFrame([
    {"metric": "pixel_AUROC", "value": anomalib_metrics.get("pixel_AUROC")},
    {"metric": "pixel_F1Score", "value": anomalib_metrics.get("pixel_F1Score")},
])

padim_pixel_df

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(padim_pixel_df["metric"], padim_pixel_df["value"], color=["#1971c2", "#2f9e44"])
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("PaDiM Pixel-Level Localization Metrics")
plt.tight_layout()

pixel_chart_path = CHART_DIR / "padim_pixel_metrics.png"
plt.savefig(pixel_chart_path, dpi=160)
plt.show()

print("Saved chart:", pixel_chart_path)

## Defect Classifier Summary

The classifier predicts specific defect type.

In [ ]:
classifier_summary = {
    "accuracy": classifier_metrics.get("accuracy"),
    "train_size": classifier_metrics.get("train_size"),
    "eval_size": classifier_metrics.get("eval_size"),
    "feature_extractor": classifier_metrics.get("feature_extractor"),
    "labels": classifier_metrics.get("labels"),
}

classifier_summary

In [ ]:
report = classifier_metrics.get("classification_report", {})
class_rows = []

for label in classifier_metrics.get("labels", []):
    values = report.get(label, {})
    class_rows.append({
        "label": label,
        "precision": values.get("precision"),
        "recall": values.get("recall"),
        "f1_score": values.get("f1-score"),
        "support": values.get("support"),
    })

classifier_report_df = pd.DataFrame(class_rows)
classifier_report_df

In [ ]:
classifier_report_df.set_index("label")[["precision", "recall", "f1_score"]].plot(kind="bar", figsize=(10, 4))
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Defect Classifier Per-Class Metrics")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()

classifier_chart_path = CHART_DIR / "defect_classifier_per_class_metrics.png"
plt.savefig(classifier_chart_path, dpi=160)
plt.show()

print("Saved chart:", classifier_chart_path)

## Severity Scoring Summary

In [ ]:
severity_csv = REPORT_DIR / "severity_scoring_examples.csv"

if severity_csv.exists():
    severity_df = pd.read_csv(severity_csv)
else:
    severity_df = pd.DataFrame()
    print("Severity examples not found. Run notebook 9 first.")

severity_df

In [ ]:
if not severity_df.empty:
    display(severity_df[["inspection_id", "defect_type", "severity_score", "severity_level", "pass_fail", "recommended_action"]])
else:
    print("No severity data available yet.")

## Final ML Pipeline Summary

In [ ]:
pipeline_summary = [
    {
        "stage": "Dataset",
        "artifact": "MVTec AD bottle",
        "status": "complete",
        "output": "292 image records + ground truth masks",
    },
    {
        "stage": "Preprocessing",
        "artifact": "ml/preprocessing.py",
        "status": "complete",
        "output": "resize, grayscale, denoise, contrast, edges",
    },
    {
        "stage": "Baseline detection",
        "artifact": "outputs/reports/baseline_detection_metrics.json",
        "status": "complete",
        "output": f"ROC-AUC {baseline_metrics.get('roc_auc')}",
    },
    {
        "stage": "Anomaly detection",
        "artifact": "models/checkpoints/padim_mvtec_bottle_v1.ckpt",
        "status": "complete",
        "output": f"Image AUROC {anomalib_metrics.get('image_AUROC')}",
    },
    {
        "stage": "Defect classification",
        "artifact": "models/defect_classifier.pkl",
        "status": "complete",
        "output": f"Accuracy {classifier_metrics.get('accuracy')}",
    },
    {
        "stage": "Severity scoring",
        "artifact": "ml/severity.py",
        "status": "complete",
        "output": "severity score, level, pass/fail, action",
    },
]

pipeline_summary_df = pd.DataFrame(pipeline_summary)
pipeline_summary_df

## Save Consolidated Evaluation Summary

In [ ]:
evaluation_summary = {
    "project": "VisionInspect AI",
    "dataset": "MVTec AD bottle",
    "baseline_metrics": baseline_metrics,
    "anomalib_padim_metrics": anomalib_metrics,
    "defect_classifier_summary": classifier_summary,
    "artifacts": artifact_rows,
    "pipeline_summary": pipeline_summary,
}

summary_json = REPORT_DIR / "ml_evaluation_summary.json"
summary_csv = REPORT_DIR / "ml_pipeline_summary.csv"
comparison_csv = REPORT_DIR / "model_comparison.csv"

with summary_json.open("w", encoding="utf-8") as file:
    json.dump(evaluation_summary, file, indent=2)

pipeline_summary_df.to_csv(summary_csv, index=False)
comparison_df.to_csv(comparison_csv, index=False)

print("Saved:", summary_json)
print("Saved:", summary_csv)
print("Saved:", comparison_csv)

## Report-Ready Conclusion

The ML phase of VisionInspect AI is complete.

Main result:

- PaDiM achieved excellent image-level anomaly detection performance.
- Pixel-level AUROC confirms useful heatmap localization.
- The defect classifier provides defect type labels for reporting and severity scoring.
- Severity scoring converts model outputs into business decisions.

The next engineering phase is to expose this through a real product workflow:

```text
FastAPI upload endpoint -> model inference -> Cloudinary heatmap -> MongoDB result -> Next.js dashboard
```